# Learn VUS curve artifact: reload and evaluate

This notebook reads the artifact created by the build notebook.

It calculates full VUS and budgeted VUS without rebuilding the threshold sweep.

Type the code marked with `# TODO`, run the cell, and inspect the result before continuing.

## 1. Imports and artifact paths

Use the repository `.venv` when launching Jupyter.

In [ ]:
from pathlib import Path
import json
import numpy as np

from src.metrics.pointwise import (
    FPR_BUDGETS,
    _compute_pr_area_from_points,
    _compute_roc_area_from_points,
    _constrained_pr_area,
    _normalised_partial_roc_area,
)

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src").is_dir(), "Launch Jupyter from the repository root."
ARTIFACT_DIR = REPO_ROOT / "notebooks" / "artifacts"
ARTIFACT_NPZ = ARTIFACT_DIR / "vus_curve_learning_demo.npz"
ARTIFACT_JSON = ARTIFACT_DIR / "vus_curve_learning_demo.json"

## 2. Load and inspect the artifact

The artifact arrays are the only input to the reducers.

In [ ]:
# TODO: load the NPZ file with allow_pickle=False.
artifact = ...

# TODO: load the JSON metadata.
metadata = ...

# TODO: assign the six stored arrays to descriptive variables.

# TODO: print the metadata and every array shape.
assert ARTIFACT_NPZ.is_file()
assert ARTIFACT_JSON.is_file()
assert set(artifact.files) == {
    "buffer_sizes",
    "thresholds",
    "precision_matrix",
    "recall_matrix",
    "false_positive_rate_matrix",
    "true_positive_rate_matrix",
}
assert metadata["fpr_budgets"] == [0.001, 0.005, 0.01]

## 3. Full VUS-PR

For each buffer row, calculate PR area from all thresholds.

Preserve the current full-VUS aggregation rule: average the finite buffer-level areas.

In [ ]:
full_pr_by_buffer = []

# TODO: loop over buffer rows.
# TODO: pass one row of precision and recall to _compute_pr_area_from_points.
# TODO: append each area.

# TODO: calculate the mean of finite areas.
full_vus_pr = ...
print(full_vus_pr)
assert np.isfinite(full_vus_pr)
assert 0.0 <= full_vus_pr <= 1.0

## 4. Full VUS-ROC

Use the same artifact and the same buffer rows, but use FPR and TPR.

In [ ]:
full_roc_by_buffer = []

# TODO: loop over buffer rows.
# TODO: pass one row of FPR and TPR to _compute_roc_area_from_points.
# TODO: append each area.

# TODO: calculate the mean of finite areas.
full_vus_roc = ...
print(full_vus_roc)
assert np.isfinite(full_vus_roc)
assert 0.0 <= full_vus_roc <= 1.0

## 5. VUS-PR@FPR-budget

Use the same PR and ROC points for each budget.

The budget changes the reducer, not the artifact.

In [ ]:
vus_pr_at_budget = {}

# TODO: loop through FPR_BUDGETS.
# TODO: for each budget, loop through buffer rows.
# TODO: call _constrained_pr_area with one row of PR values and FPR values.
# TODO: aggregate the buffer-level values using the current budgeted-VUS rule.
# TODO: store one result under str(budget).

print(vus_pr_at_budget)
assert set(vus_pr_at_budget) == {str(budget) for budget in FPR_BUDGETS}
assert all(np.isfinite(value) and 0.0 <= value <= 1.0 for value in vus_pr_at_budget.values())

## 6. VUS-ROC@FPR-budget

For ROC, use partial area up to the selected FPR budget and normalize by that budget.

In [ ]:
vus_roc_at_budget = {}

# TODO: loop through FPR_BUDGETS.
# TODO: for each budget, loop through buffer rows.
# TODO: call _normalised_partial_roc_area with one row of FPR and TPR values.
# TODO: aggregate the buffer-level values using the current budgeted-VUS rule.
# TODO: store one result under str(budget).

print(vus_roc_at_budget)
assert set(vus_roc_at_budget) == {str(budget) for budget in FPR_BUDGETS}
assert all(np.isfinite(value) and 0.0 <= value <= 1.0 for value in vus_roc_at_budget.values())

## 7. Final invariants

These checks confirm that the reducers are isolated from artifact creation.

In [ ]:
# TODO: assert that both budget dictionaries contain exactly the three locked budgets.
# TODO: assert that every finite metric lies in [0.0, 1.0].
# TODO: copy the artifact arrays, recompute with a different budget, and assert the arrays are unchanged.
# TODO: print a compact summary of full and budgeted metrics.
assert set(vus_pr_at_budget) == {str(budget) for budget in FPR_BUDGETS}
assert set(vus_roc_at_budget) == {str(budget) for budget in FPR_BUDGETS}
assert np.isfinite(full_vus_pr) and np.isfinite(full_vus_roc)